# WiDS 2026 Breakthrough Internal Survival Ensemble


In [1]:
import os
import json
import math
import random
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb


SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")


def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26/wiDSworldwide_globaldathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if all(os.path.exists(os.path.join(path, f)) for f in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path
    if os.path.isdir("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            file_set = set(files)
            if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(file_set):
                return root
    raise FileNotFoundError("Could not locate train.csv, test.csv, sample_submission.csv")


def seed_everything(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r["dist_min_ci_0_5h"].clip(lower=1.0)
    speed = r["closing_speed_m_per_h"]
    perims = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"].clip(lower=0.0)
    area_growth = r["area_growth_rate_ha_per_h"]
    radial = r["radial_growth_rate_m_per_h"].clip(lower=0.0)
    align = r["alignment_abs"]
    fire_radius = np.sqrt(np.maximum(area_first, 0.0) * 10000.0 / np.pi)
    closing_pos = speed.clip(lower=0.0)
    eff_close = closing_pos + radial

    r["dist_km"] = dist / 1000.0
    r["log_distance"] = np.log1p(dist)
    r["inv_distance"] = 1.0 / (r["dist_km"] + 0.1)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["sqrt_distance"] = np.sqrt(dist)
    r["dist_km_sq"] = r["dist_km"] ** 2
    r["dist_km_cb"] = r["dist_km"] ** 3

    r["fire_radius_km"] = fire_radius / 1000.0
    r["radius_to_dist"] = fire_radius / dist
    r["edge_gap_km"] = np.maximum(dist - fire_radius - 5000.0, -5000.0) / 1000.0
    r["area_to_dist_ratio"] = area_first / (r["dist_km"] + 0.1)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    r["has_movement"] = (perims > 1).astype(float)
    r["effective_closing_speed"] = eff_close
    r["eta_hours"] = np.where(closing_pos > 1e-3, dist / np.maximum(closing_pos, 1e-3), 9999.0).clip(max=9999.0)
    r["eta_effective"] = np.where(eff_close > 1e-3, dist / np.maximum(eff_close, 1e-3), 9999.0).clip(max=9999.0)
    r["edge_eta_effective"] = np.where(
        eff_close > 1e-3,
        np.maximum(dist - fire_radius - 5000.0, 0.0) / np.maximum(eff_close, 1e-3),
        9999.0,
    ).clip(max=9999.0)
    r["log_eta"] = np.log1p(r["eta_hours"])
    r["log_eta_effective"] = np.log1p(r["eta_effective"])
    r["log_edge_eta_effective"] = np.log1p(r["edge_eta_effective"])

    for h in [12, 24, 48, 72]:
        r[f"gap_h{h}_km"] = (dist - fire_radius - 5000.0 - eff_close * h) / 1000.0
        r[f"threat_h{h}"] = (fire_radius + 5000.0 + eff_close * h - dist) / 1000.0
        r[f"sig_h{h}"] = 1.0 / (1.0 + np.exp(np.clip(r[f"gap_h{h}_km"], -40.0, 40.0)))

    r["threat_score"] = align * closing_pos / np.log1p(dist)
    r["threat_score_eff"] = align * eff_close / np.log1p(dist)
    r["threat_score_sq"] = r["threat_score_eff"] ** 2
    r["fire_urgency"] = perims * closing_pos
    r["growth_intensity"] = area_growth * perims
    r["speed_alignment"] = closing_pos * align
    r["effective_alignment"] = eff_close * align

    r["projected_advance_pos"] = np.maximum(r["projected_advance_m"], 0.0)
    r["closing_distance_pos"] = np.maximum(-r["dist_change_ci_0_5h"], 0.0)
    r["slope_toward"] = np.maximum(-r["dist_slope_ci_0_5h"], 0.0)
    r["accel_toward"] = np.maximum(-r["dist_accel_m_per_h2"], 0.0)

    r["zone_3km"] = (dist < 3000.0).astype(float)
    r["zone_near"] = (dist < 5000.0).astype(float)
    r["zone_warning"] = ((dist >= 5000.0) & (dist < 10000.0)).astype(float)
    r["zone_far"] = (dist >= 10000.0).astype(float)
    r["zone_20km"] = (dist < 20000.0).astype(float)

    r["is_summer"] = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_night"] = ((r["event_start_hour"] <= 6) | (r["event_start_hour"] >= 21)).astype(float)

    r["hour_sin"] = np.sin(2.0 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2.0 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2.0 * np.pi * (r["event_start_month"] - 1.0) / 12.0)
    r["month_cos"] = np.cos(2.0 * np.pi * (r["event_start_month"] - 1.0) / 12.0)
    r["dow_sin"] = np.sin(2.0 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2.0 * np.pi * r["event_start_dayofweek"] / 7.0)

    def pct_rank(values, ref_values):
        ref_values = np.asarray(ref_values, dtype=float)
        if ref_values.size == 0:
            return np.zeros(len(values), dtype=float)
        ref_sorted = np.sort(ref_values)
        return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side="right") / ref_sorted.size

    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1.0).values
    ref_eff = (
        ref["closing_speed_m_per_h"].clip(lower=0.0).values
        + ref["radial_growth_rate_m_per_h"].clip(lower=0.0).values
    )
    ref_threat = ref["alignment_abs"].values * ref_eff / np.log1p(ref["dist_min_ci_0_5h"].clip(lower=1.0).values)

    r["dist_rank_global"] = pct_rank(dist.values, ref_dist)
    r["eff_rank_global"] = pct_rank(eff_close.values, ref_eff)
    r["threat_rank_global"] = pct_rank(r["threat_score_eff"].values, ref_threat)

    ref_near = ref["dist_min_ci_0_5h"].clip(lower=1.0).values < 5000.0
    cur_near = dist.values < 5000.0
    near_speed_rank = np.zeros(len(r), dtype=float)
    far_threat_rank = np.zeros(len(r), dtype=float)
    near_ref_eff = ref_eff[ref_near]
    far_ref_threat = ref_threat[~ref_near]
    if cur_near.any():
        near_speed_rank[cur_near] = pct_rank(eff_close.values[cur_near], near_ref_eff)
    if (~cur_near).any():
        far_threat_rank[~cur_near] = pct_rank(r["threat_score_eff"].values[~cur_near], far_ref_threat)
    r["near_speed_rank"] = near_speed_rank
    r["far_threat_rank"] = far_threat_rank

    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    r = r.drop(columns=[c for c in drop_cols if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return r


def compute_c_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    risk = np.asarray(risk, dtype=float)
    n = len(time)
    concordant = 0.0
    comparable = 0.0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comparable += 1.0
            if risk[i] > risk[j]:
                concordant += 1.0
            elif risk[i] == risk[j]:
                concordant += 0.5
    return concordant / comparable if comparable else 0.5


def compute_brier(time, event, prob, horizon):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    prob = np.asarray(prob, dtype=float)
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    return float(np.mean((np.clip(prob[valid], 0.0, 1.0) - y_true) ** 2))


def compute_hybrid_score(time, event, p24, p48, p72):
    risk = 0.3 * np.asarray(p24, dtype=float) + 0.4 * np.asarray(p48, dtype=float) + 0.3 * np.asarray(p72, dtype=float)
    c_idx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24)
    b48 = compute_brier(time, event, p48, 48)
    b72 = compute_brier(time, event, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return {
        "hybrid": float(0.3 * c_idx + 0.7 * (1.0 - wb)),
        "c_index": float(c_idx),
        "weighted_brier": float(wb),
        "brier24": float(b24),
        "brier48": float(b48),
        "brier72": float(b72),
    }


def enforce_monotonicity(preds):
    preds = np.clip(np.asarray(preds, dtype=float), 0.0, 1.0)
    out = preds.copy()
    for j in range(1, out.shape[1]):
        out[:, j] = np.maximum(out[:, j], out[:, j - 1])
    return np.clip(out, 0.0, 1.0)


def validate_submission(sub, sample):
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == required
    assert len(sub) == len(sample)
    assert sub["event_id"].equals(sample["event_id"])
    assert sub["event_id"].nunique() == len(sub)
    vals = sub[required[1:]].values
    assert np.isfinite(vals).all()
    assert ((vals >= 0.0) & (vals <= 1.0)).all()
    assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
    assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
    assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)


def make_binary_target(time_vals, event_vals, horizon):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown


def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    for i, t in enumerate(unique_t):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            surv[i] = 1.0 - cens / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]

    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0

    weights = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
        else:
            weights[i] = 0.0
    return weights


def safe_n_splits(y, desired=5):
    y = np.asarray(y)
    if len(y) < 4:
        return 0
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    if pos == 0 or neg == 0:
        return 0
    return max(2, min(desired, pos, neg))


def constant_pred(y, n):
    y = np.asarray(y, dtype=float)
    val = float(y.mean()) if len(y) else 0.5
    return np.full(n, val, dtype=float)


def fit_with_optional_weight(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model
    if isinstance(model, Pipeline):
        last_name = model.steps[-1][0]
        try:
            model.fit(X, y, **{f"{last_name}__sample_weight": sample_weight})
            return model
        except Exception:
            model.fit(X, y)
            return model
    try:
        model.fit(X, y, sample_weight=sample_weight)
        return model
    except Exception:
        model.fit(X, y)
        return model


def safe_predict_proba(model, X):
    p = model.predict_proba(X)
    p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else p


def soft_gate(dist_m, center, width):
    z = np.clip((np.asarray(dist_m, dtype=float) - center) / max(width, 1.0), -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(z))


def stabilize(pred, anchor, alpha):
    pred = np.asarray(pred, dtype=float)
    anchor = np.asarray(anchor, dtype=float)
    return np.clip(alpha * pred + (1.0 - alpha) * anchor, 0.0, 1.0)


def repeated_isotonic(train_signal, test_signal, y, mask, seeds, desired_splits=5):
    train_signal = np.asarray(train_signal, dtype=float)
    test_signal = np.asarray(test_signal, dtype=float)
    y = np.asarray(y, dtype=int)
    mask = np.asarray(mask, dtype=bool)

    valid_idx = np.where(mask)[0]
    if len(valid_idx) == 0:
        base = np.full(len(train_signal), 0.5, dtype=float)
        return base, np.full(len(test_signal), 0.5, dtype=float)

    yv = y[valid_idx]
    if len(np.unique(yv)) < 2:
        base_val = float(yv.mean())
        return np.full(len(train_signal), base_val, dtype=float), np.full(len(test_signal), base_val, dtype=float)

    n_splits = safe_n_splits(yv, desired_splits)
    if n_splits == 0:
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(train_signal[valid_idx], yv)
        full_train = ir.predict(train_signal)
        full_test = ir.predict(test_signal)
        return np.clip(full_train, 0.0, 1.0), np.clip(full_test, 0.0, 1.0)

    oof = np.zeros(len(train_signal), dtype=float)
    cnt = np.zeros(len(train_signal), dtype=float)
    full_fill = np.zeros(len(train_signal), dtype=float)
    test_pred = np.zeros(len(test_signal), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(valid_idx, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            ir = IsotonicRegression(out_of_bounds="clip")
            ir.fit(train_signal[tr_idx], y[tr_idx])
            oof[va_idx] += ir.predict(train_signal[va_idx])
            cnt[va_idx] += 1.0

        ir_full = IsotonicRegression(out_of_bounds="clip")
        ir_full.fit(train_signal[valid_idx], yv)
        full_fill += ir_full.predict(train_signal)
        test_pred += ir_full.predict(test_signal)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


def repeated_binary_model(X_train, X_test, y, mask, build_model, seeds, desired_splits=5, horizon=None, use_ipcw=False):
    X_train = X_train.copy()
    X_test = X_test.copy()
    y = np.asarray(y, dtype=int)
    mask = np.asarray(mask, dtype=bool)

    valid_idx = np.where(mask)[0]
    if len(valid_idx) == 0:
        return np.full(len(X_train), 0.5, dtype=float), np.full(len(X_test), 0.5, dtype=float)

    Xv = X_train.iloc[valid_idx]
    yv = y[valid_idx]
    if len(np.unique(yv)) < 2:
        base_val = float(yv.mean())
        return np.full(len(X_train), base_val, dtype=float), np.full(len(X_test), base_val, dtype=float)

    n_splits = safe_n_splits(yv, desired_splits)
    if n_splits == 0:
        model_full = build_model(seeds[0])
        sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon) if (use_ipcw and horizon is not None) else None
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        full_train = safe_predict_proba(model_full, X_train)
        full_test = safe_predict_proba(model_full, X_test)
        return np.clip(full_train, 0.0, 1.0), np.clip(full_test, 0.0, 1.0)

    oof = np.zeros(len(X_train), dtype=float)
    cnt = np.zeros(len(X_train), dtype=float)
    full_fill = np.zeros(len(X_train), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            model = build_model(seed)
            sw = None
            if use_ipcw and horizon is not None:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        sw_full = None
        if use_ipcw and horizon is not None:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        full_fill += safe_predict_proba(model_full, X_train)
        test_pred += safe_predict_proba(model_full, X_test)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


def make_lgb_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(dict(objective="binary", random_state=seed, n_jobs=-1, verbose=-1))
        return lgb.LGBMClassifier(**p)
    return _build


def make_cat_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(
            dict(
                loss_function="Logloss",
                eval_metric="Logloss",
                random_seed=seed,
                verbose=False,
                allow_writing_files=False,
                thread_count=-1,
            )
        )
        return CatBoostClassifier(**p)
    return _build


def make_logit_builder(C):
    def _build(seed):
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                ("lr", LogisticRegression(C=C, penalty="l2", solver="lbfgs", max_iter=4000)),
            ]
        )
    return _build


def repeated_xgb_cox_risk(X_train, X_test, time, event, seeds, cfgs, fallback_train, fallback_test, desired_splits=5):
    X_train = X_train.copy()
    X_test = X_test.copy()
    time = np.asarray(time, dtype=np.float32)
    event = np.asarray(event, dtype=int)
    fallback_train = np.asarray(fallback_train, dtype=float)
    fallback_test = np.asarray(fallback_test, dtype=float)

    n = len(X_train)
    m = len(X_test)
    n_splits = safe_n_splits(event, desired_splits)

    oof = np.zeros(n, dtype=float)
    cnt = np.zeros(n, dtype=float)
    full_fill = np.zeros(n, dtype=float)
    test_pred = np.zeros(m, dtype=float)

    total_models = max(1, len(seeds) * len(cfgs))

    for cfg in cfgs:
        for seed in seeds:
            if n_splits == 0:
                oof += fallback_train / total_models
                cnt += 1.0 / total_models
            else:
                cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
                for tr_idx, va_idx in cv.split(X_train, event):
                    model = xgb.XGBRegressor(
                        objective="survival:cox",
                        n_estimators=cfg["n_estimators"],
                        learning_rate=cfg["learning_rate"],
                        max_depth=cfg["max_depth"],
                        min_child_weight=cfg["min_child_weight"],
                        subsample=cfg["subsample"],
                        colsample_bytree=cfg["colsample_bytree"],
                        reg_alpha=cfg["reg_alpha"],
                        reg_lambda=cfg["reg_lambda"],
                        gamma=cfg.get("gamma", 0.0),
                        tree_method="hist",
                        n_jobs=-1,
                        random_state=seed,
                    )
                    y_tr = time[tr_idx].copy()
                    y_tr[event[tr_idx] == 0] *= -1.0
                    try:
                        model.fit(X_train.iloc[tr_idx], y_tr)
                        oof[va_idx] += np.asarray(model.predict(X_train.iloc[va_idx]), dtype=float)
                        cnt[va_idx] += 1.0
                    except Exception:
                        oof[va_idx] += fallback_train[va_idx]
                        cnt[va_idx] += 1.0

            model_full = xgb.XGBRegressor(
                objective="survival:cox",
                n_estimators=cfg["n_estimators"],
                learning_rate=cfg["learning_rate"],
                max_depth=cfg["max_depth"],
                min_child_weight=cfg["min_child_weight"],
                subsample=cfg["subsample"],
                colsample_bytree=cfg["colsample_bytree"],
                reg_alpha=cfg["reg_alpha"],
                reg_lambda=cfg["reg_lambda"],
                gamma=cfg.get("gamma", 0.0),
                tree_method="hist",
                n_jobs=-1,
                random_state=seed,
            )
            y_full = time.copy()
            y_full[event == 0] *= -1.0
            try:
                model_full.fit(X_train, y_full)
                full_fill += np.asarray(model_full.predict(X_train), dtype=float)
                test_pred += np.asarray(model_full.predict(X_test), dtype=float)
            except Exception:
                full_fill += fallback_train
                test_pred += fallback_test

    full_fill /= total_models
    test_pred /= total_models
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.asarray(oof, dtype=float), np.asarray(test_pred, dtype=float)


def make_aft_dmatrix(X, lower, upper):
    dmat = xgb.DMatrix(X.values, feature_names=list(X.columns))
    dmat.set_float_info("label_lower_bound", np.asarray(lower, dtype=np.float32))
    dmat.set_float_info("label_upper_bound", np.asarray(upper, dtype=np.float32))
    return dmat


def repeated_xgb_aft_risk(X_train, X_test, time, event, seeds, cfgs, fallback_train, fallback_test, desired_splits=5):
    X_train = X_train.copy()
    X_test = X_test.copy()
    time = np.asarray(time, dtype=np.float32)
    event = np.asarray(event, dtype=int)
    lower_full = time.copy()
    upper_full = np.where(event == 1, time, np.inf).astype(np.float32)

    fallback_train = np.asarray(fallback_train, dtype=float)
    fallback_test = np.asarray(fallback_test, dtype=float)

    n = len(X_train)
    m = len(X_test)
    n_splits = safe_n_splits(event, desired_splits)

    oof = np.zeros(n, dtype=float)
    cnt = np.zeros(n, dtype=float)
    full_fill = np.zeros(n, dtype=float)
    test_pred = np.zeros(m, dtype=float)

    total_models = max(1, len(seeds) * len(cfgs))
    dtest = xgb.DMatrix(X_test.values, feature_names=list(X_test.columns))

    for cfg in cfgs:
        for seed in seeds:
            params = {
                "objective": "survival:aft",
                "eval_metric": "aft-nloglik",
                "aft_loss_distribution": cfg.get("aft_loss_distribution", "normal"),
                "aft_loss_distribution_scale": cfg.get("aft_loss_distribution_scale", 1.0),
                "eta": cfg["learning_rate"],
                "max_depth": cfg["max_depth"],
                "min_child_weight": cfg["min_child_weight"],
                "subsample": cfg["subsample"],
                "colsample_bytree": cfg["colsample_bytree"],
                "alpha": cfg["reg_alpha"],
                "lambda": cfg["reg_lambda"],
                "gamma": cfg.get("gamma", 0.0),
                "tree_method": "hist",
                "nthread": -1,
                "seed": seed,
            }

            if n_splits == 0:
                oof += fallback_train / total_models
                cnt += 1.0 / total_models
            else:
                cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
                for tr_idx, va_idx in cv.split(X_train, event):
                    try:
                        dtr = make_aft_dmatrix(X_train.iloc[tr_idx], lower_full[tr_idx], upper_full[tr_idx])
                        dva = xgb.DMatrix(X_train.iloc[va_idx].values, feature_names=list(X_train.columns))
                        bst = xgb.train(params, dtr, num_boost_round=cfg["num_boost_round"], verbose_eval=False)
                        pred_va = np.asarray(bst.predict(dva), dtype=float)
                        oof[va_idx] += -pred_va
                        cnt[va_idx] += 1.0
                    except Exception:
                        oof[va_idx] += fallback_train[va_idx]
                        cnt[va_idx] += 1.0

            try:
                dtrain_full = make_aft_dmatrix(X_train, lower_full, upper_full)
                bst_full = xgb.train(params, dtrain_full, num_boost_round=cfg["num_boost_round"], verbose_eval=False)
                dtrain_pred = xgb.DMatrix(X_train.values, feature_names=list(X_train.columns))
                full_fill += -np.asarray(bst_full.predict(dtrain_pred), dtype=float)
                test_pred += -np.asarray(bst_full.predict(dtest), dtype=float)
            except Exception:
                full_fill += fallback_train
                test_pred += fallback_test

    full_fill /= total_models
    test_pred /= total_models
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.asarray(oof, dtype=float), np.asarray(test_pred, dtype=float)


def normalize_rank(x):
    x = np.asarray(x, dtype=float)
    if len(x) <= 1:
        return np.zeros(len(x), dtype=float)
    order = np.argsort(np.argsort(x, kind="mergesort"), kind="mergesort")
    return order / (len(x) - 1.0)


def rank_to_distribution(base_vals, rank_signal):
    base_vals = np.asarray(base_vals, dtype=float)
    rank_signal = np.asarray(rank_signal, dtype=float)
    order = np.argsort(rank_signal, kind="mergesort")
    sorted_base = np.sort(base_vals)
    out = np.empty_like(sorted_base)
    out[order] = sorted_base
    return out


STRUCT_PRIOR = {
    "anchor": 1.15,
    "phys": 1.10,
    "zone_lgb": 1.05,
    "glob_lgb": 1.00,
    "glob_cat": 0.95,
    "lin": 0.85,
    "cox": 1.00,
    "aft": 1.00,
}


def build_data_prior(P, y, names, mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.sum() == 0:
        w = np.array([STRUCT_PRIOR.get(n, 1.0) for n in names], dtype=float)
        w = np.clip(w, 1e-8, None)
        return w / w.sum()

    losses = []
    for j, name in enumerate(names):
        pred = np.clip(P[:, j], 1e-6, 1.0 - 1e-6)
        loss = float(np.mean((pred[mask] - y[mask]) ** 2))
        losses.append(loss)

    losses = np.asarray(losses, dtype=float)
    spread = max(float(np.std(losses)), 0.01)
    data_weight = np.exp(-(losses - losses.min()) / spread)
    struct_weight = np.array([STRUCT_PRIOR.get(name, 1.0) for name in names], dtype=float)
    w = np.clip(data_weight * struct_weight, 1e-8, None)
    return w / w.sum()


def optimize_blend(P, y, prior, reg, shrink):
    P = np.asarray(P, dtype=float)
    y = np.asarray(y, dtype=float)
    prior = np.asarray(prior, dtype=float)
    prior = np.clip(prior, 1e-8, None)
    prior = prior / prior.sum()

    if P.shape[1] == 1:
        return prior.copy(), prior.copy()

    def objective(w):
        pred = P @ w
        return np.mean((pred - y) ** 2) + reg * np.sum((w - prior) ** 2)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 1.0)] * P.shape[1]

    try:
        res = minimize(
            objective,
            prior.copy(),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 400, "ftol": 1e-10},
        )
        if res.success:
            opt = np.clip(np.asarray(res.x, dtype=float), 0.0, None)
            opt = opt / opt.sum()
        else:
            opt = prior.copy()
    except Exception:
        opt = prior.copy()

    final = shrink * opt + (1.0 - shrink) * prior
    final = np.clip(final, 1e-8, None)
    final = final / final.sum()
    return final, opt


def zone_blend_cfg(h, zone):
    if zone == "near":
        if h == 12:
            return 0.10, 0.35
        if h == 24:
            return 0.09, 0.32
        return 0.08, 0.30
    if h == 12:
        return 0.08, 0.50
    if h == 24:
        return 0.07, 0.45
    return 0.06, 0.42


seed_everything(SEED)
DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("DATA_DIR =", DATA_DIR)
print("TRAIN =", train_df.shape, "TEST =", test_df.shape)

train_proc = create_features(train_df, fit_df=train_df)
test_proc = create_features(test_df, fit_df=train_df)

time_values = train_df["time_to_hit_hours"].astype(float).values
event_values = train_df["event"].astype(int).values
dist_train = train_df["dist_min_ci_0_5h"].astype(float).values
dist_test = test_df["dist_min_ci_0_5h"].astype(float).values

RAW_FEATURES = [c for c in train_df.columns if c not in ["event_id", "event", "time_to_hit_hours"]]

NEAR_LGB_FEATURES = [
    "closing_speed_m_per_h",
    "radial_growth_rate_m_per_h",
    "alignment_abs",
    "num_perimeters_0_5h",
    "area_growth_rate_ha_per_h",
    "eta_effective",
    "log_eta_effective",
    "dist_km",
    "threat_score_eff",
    "near_speed_rank",
    "event_start_hour",
    "is_afternoon",
    "fire_urgency",
    "area_first_ha",
    "fire_radius_km",
    "low_temporal_resolution_0_5h",
    "dt_first_last_0_5h",
    "edge_gap_km",
    "log_edge_eta_effective",
    "sig_h12",
    "sig_h24",
    "sig_h48",
]
NEAR_LGB_FEATURES = [c for c in NEAR_LGB_FEATURES if c in train_proc.columns]

FAR_LGB_FEATURES = [
    "dist_km",
    "log_distance",
    "inv_distance",
    "closing_speed_m_per_h",
    "alignment_abs",
    "threat_score_eff",
    "log_eta_effective",
    "eta_effective",
    "area_to_dist_ratio",
    "num_perimeters_0_5h",
    "far_threat_rank",
    "is_summer",
    "zone_far",
    "low_temporal_resolution_0_5h",
    "dt_first_last_0_5h",
    "log1p_growth",
    "dist_fit_r2_0_5h",
    "edge_gap_km",
    "log_edge_eta_effective",
    "sig_h24",
    "sig_h48",
]
FAR_LGB_FEATURES = [c for c in FAR_LGB_FEATURES if c in train_proc.columns]

GLOBAL_TREE_FEATURES = sorted(
    set(
        NEAR_LGB_FEATURES
        + FAR_LGB_FEATURES
        + [
            "inv_distance_sq",
            "sqrt_distance",
            "threat_score",
            "growth_intensity",
            "speed_alignment",
            "effective_alignment",
            "zone_warning",
            "hour_sin",
            "hour_cos",
            "month_sin",
            "month_cos",
            "dow_sin",
            "dow_cos",
            "projected_advance_pos",
            "closing_distance_pos",
            "slope_toward",
            "log_area_dist_ratio",
            "log1p_area_first",
            "dist_rank_global",
            "eff_rank_global",
            "threat_rank_global",
            "gap_h12_km",
            "gap_h24_km",
            "gap_h48_km",
            "threat_h12",
            "threat_h24",
            "threat_h48",
        ]
    )
)
GLOBAL_TREE_FEATURES = [c for c in GLOBAL_TREE_FEATURES if c in train_proc.columns]

LINEAR_FEATURES = [
    "dist_km",
    "log_distance",
    "inv_distance",
    "closing_speed_m_per_h",
    "radial_growth_rate_m_per_h",
    "alignment_abs",
    "threat_score_eff",
    "log_eta_effective",
    "eta_effective",
    "area_to_dist_ratio",
    "fire_radius_km",
    "num_perimeters_0_5h",
    "has_movement",
    "near_speed_rank",
    "far_threat_rank",
    "low_temporal_resolution_0_5h",
    "dt_first_last_0_5h",
    "is_summer",
    "is_afternoon",
    "hour_sin",
    "hour_cos",
    "zone_near",
    "zone_far",
    "edge_gap_km",
    "sig_h12",
    "sig_h24",
    "sig_h48",
    "gap_h12_km",
    "gap_h24_km",
    "gap_h48_km",
]
LINEAR_FEATURES = [c for c in LINEAR_FEATURES if c in train_proc.columns]

SURV_FEATURES = sorted(
    set(
        GLOBAL_TREE_FEATURES
        + [
            "dist_km_cb",
            "fire_radius_km",
            "edge_gap_km",
            "log_edge_eta_effective",
            "zone_3km",
            "zone_near",
            "zone_warning",
            "zone_20km",
            "gap_h12_km",
            "gap_h24_km",
            "gap_h48_km",
            "threat_h12",
            "threat_h24",
            "threat_h48",
            "sig_h12",
            "sig_h24",
            "sig_h48",
            "accel_toward",
            "radius_to_dist",
            "threat_score_sq",
        ]
    )
)
SURV_FEATURES = [c for c in SURV_FEATURES if c in train_proc.columns]

X_global_train = train_proc[GLOBAL_TREE_FEATURES].astype(np.float32)
X_global_test = test_proc[GLOBAL_TREE_FEATURES].astype(np.float32)
X_near_lgb_train = train_proc[NEAR_LGB_FEATURES].astype(np.float32)
X_near_lgb_test = test_proc[NEAR_LGB_FEATURES].astype(np.float32)
X_far_lgb_train = train_proc[FAR_LGB_FEATURES].astype(np.float32)
X_far_lgb_test = test_proc[FAR_LGB_FEATURES].astype(np.float32)
X_linear_train = train_proc[LINEAR_FEATURES].astype(np.float32)
X_linear_test = test_proc[LINEAR_FEATURES].astype(np.float32)
X_surv_train = train_proc[SURV_FEATURES].astype(np.float32)
X_surv_test = test_proc[SURV_FEATURES].astype(np.float32)

y_map = {}
mask_map = {}
for h in [12, 24, 48]:
    y_map[h], mask_map[h] = make_binary_target(time_values, event_values, h)

ANCHOR_SEEDS = (123, 456, 789, 2024, 2025)
TREE_SEEDS = (123, 456, 789, 777, 666)
CAT_SEEDS = (123, 456, 789)
LINEAR_SEEDS = (123, 456, 789, 2024)
SURV_SEEDS = (123, 456, 789, 2024, 2025)

global_lgb_cfgs = {
    12: {
        "max_depth": 2,
        "learning_rate": 0.040,
        "n_estimators": 220,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "min_child_samples": 4,
        "reg_alpha": 0.5,
        "reg_lambda": 1.5,
        "num_leaves": 4,
    },
    24: {
        "max_depth": 2,
        "learning_rate": 0.035,
        "n_estimators": 260,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "min_child_samples": 6,
        "reg_alpha": 0.8,
        "reg_lambda": 2.5,
        "num_leaves": 4,
    },
    48: {
        "max_depth": 3,
        "learning_rate": 0.030,
        "n_estimators": 260,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_samples": 8,
        "reg_alpha": 1.0,
        "reg_lambda": 3.0,
        "num_leaves": 6,
    },
}

global_cat_cfgs = {
    12: {
        "iterations": 280,
        "learning_rate": 0.030,
        "depth": 4,
        "l2_leaf_reg": 10.0,
        "bootstrap_type": "Bernoulli",
        "subsample": 0.85,
        "random_strength": 0.8,
    },
    24: {
        "iterations": 320,
        "learning_rate": 0.030,
        "depth": 4,
        "l2_leaf_reg": 12.0,
        "bootstrap_type": "Bernoulli",
        "subsample": 0.85,
        "random_strength": 0.8,
    },
    48: {
        "iterations": 340,
        "learning_rate": 0.028,
        "depth": 4,
        "l2_leaf_reg": 14.0,
        "bootstrap_type": "Bernoulli",
        "subsample": 0.85,
        "random_strength": 0.8,
    },
}

near_lgb_cfgs = {
    12: {
        "max_depth": 2,
        "learning_rate": 0.040,
        "n_estimators": 220,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "min_child_samples": 4,
        "reg_alpha": 0.5,
        "reg_lambda": 1.5,
        "num_leaves": 4,
    },
    24: {
        "max_depth": 2,
        "learning_rate": 0.040,
        "n_estimators": 200,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "min_child_samples": 4,
        "reg_alpha": 0.5,
        "reg_lambda": 1.5,
        "num_leaves": 4,
    },
    48: {
        "max_depth": 2,
        "learning_rate": 0.050,
        "n_estimators": 150,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "min_child_samples": 3,
        "reg_alpha": 0.3,
        "reg_lambda": 1.0,
        "num_leaves": 4,
    },
}

far_lgb_cfgs = {
    24: {
        "max_depth": 2,
        "learning_rate": 0.030,
        "n_estimators": 200,
        "subsample": 0.70,
        "colsample_bytree": 0.70,
        "min_child_samples": 8,
        "reg_alpha": 1.0,
        "reg_lambda": 3.0,
        "num_leaves": 4,
    },
    48: {
        "max_depth": 2,
        "learning_rate": 0.050,
        "n_estimators": 150,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "min_child_samples": 6,
        "reg_alpha": 0.5,
        "reg_lambda": 2.0,
        "num_leaves": 4,
    },
}

linear_cs = {12: 0.60, 24: 0.80, 48: 1.00}

cox_cfgs = [
    {
        "n_estimators": 260,
        "learning_rate": 0.035,
        "max_depth": 2,
        "min_child_weight": 6,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.5,
        "reg_lambda": 2.0,
    },
    {
        "n_estimators": 340,
        "learning_rate": 0.025,
        "max_depth": 3,
        "min_child_weight": 8,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.8,
        "reg_lambda": 3.0,
    },
]

aft_cfgs = [
    {
        "num_boost_round": 260,
        "learning_rate": 0.035,
        "max_depth": 2,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.5,
        "reg_lambda": 2.0,
        "aft_loss_distribution": "normal",
        "aft_loss_distribution_scale": 1.0,
    },
    {
        "num_boost_round": 340,
        "learning_rate": 0.025,
        "max_depth": 3,
        "min_child_weight": 6,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.8,
        "reg_lambda": 3.0,
        "aft_loss_distribution": "normal",
        "aft_loss_distribution_scale": 1.2,
    },
]

anchor_signal_train = 1.0 / (train_proc["dist_km"].values + 0.25)
anchor_signal_test = 1.0 / (test_proc["dist_km"].values + 0.25)

anchor_oof = {}
anchor_test = {}
phys_oof = {}
phys_test = {}

for h in [12, 24, 48]:
    anchor_oof[h], anchor_test[h] = repeated_isotonic(
        anchor_signal_train,
        anchor_signal_test,
        y_map[h],
        mask_map[h],
        ANCHOR_SEEDS,
        desired_splits=5,
    )
    phys_signal_train = -np.asarray(train_proc[f"gap_h{h}_km"].values, dtype=float)
    phys_signal_test = -np.asarray(test_proc[f"gap_h{h}_km"].values, dtype=float)
    phys_oof[h], phys_test[h] = repeated_isotonic(
        phys_signal_train,
        phys_signal_test,
        y_map[h],
        mask_map[h],
        ANCHOR_SEEDS,
        desired_splits=5,
    )
    phys_oof[h] = stabilize(phys_oof[h], anchor_oof[h], 0.96 if h == 48 else 0.94)
    phys_test[h] = stabilize(phys_test[h], anchor_test[h], 0.96 if h == 48 else 0.94)
    print(f"anchor@{h}: {compute_brier(time_values, event_values, anchor_oof[h], h):.6f}")
    print(f"phys@{h}:   {compute_brier(time_values, event_values, phys_oof[h], h):.6f}")

global_lgb_oof, global_lgb_test = {}, {}
global_cat_oof, global_cat_test = {}, {}
linear_oof, linear_test = {}, {}
near_lgb_oof, near_lgb_test = {}, {}
far_lgb_oof, far_lgb_test = {}, {}

for h in [12, 24, 48]:
    global_lgb_oof[h], global_lgb_test[h] = repeated_binary_model(
        X_global_train,
        X_global_test,
        y_map[h],
        mask_map[h],
        make_lgb_builder(global_lgb_cfgs[h]),
        TREE_SEEDS,
        desired_splits=5,
        horizon=h,
        use_ipcw=(h in [24, 48]),
    )
    global_lgb_oof[h] = stabilize(global_lgb_oof[h], anchor_oof[h], 0.92)
    global_lgb_test[h] = stabilize(global_lgb_test[h], anchor_test[h], 0.92)

    global_cat_oof[h], global_cat_test[h] = repeated_binary_model(
        X_global_train,
        X_global_test,
        y_map[h],
        mask_map[h],
        make_cat_builder(global_cat_cfgs[h]),
        CAT_SEEDS,
        desired_splits=5,
        horizon=h,
        use_ipcw=(h in [24, 48]),
    )
    global_cat_oof[h] = stabilize(global_cat_oof[h], anchor_oof[h], 0.90)
    global_cat_test[h] = stabilize(global_cat_test[h], anchor_test[h], 0.90)

    linear_oof[h], linear_test[h] = repeated_binary_model(
        X_linear_train,
        X_linear_test,
        y_map[h],
        mask_map[h],
        make_logit_builder(linear_cs[h]),
        LINEAR_SEEDS,
        desired_splits=5,
        horizon=h,
        use_ipcw=(h in [24, 48]),
    )
    linear_oof[h] = stabilize(linear_oof[h], anchor_oof[h], 0.88)
    linear_test[h] = stabilize(linear_test[h], anchor_test[h], 0.88)

    near_lgb_oof[h], near_lgb_test[h] = repeated_binary_model(
        X_near_lgb_train,
        X_near_lgb_test,
        y_map[h],
        mask_map[h],
        make_lgb_builder(near_lgb_cfgs[h]),
        TREE_SEEDS,
        desired_splits=5,
        horizon=h,
        use_ipcw=(h in [24, 48]),
    )
    near_lgb_oof[h] = stabilize(near_lgb_oof[h], anchor_oof[h], 0.94)
    near_lgb_test[h] = stabilize(near_lgb_test[h], anchor_test[h], 0.94)

    print(f"glob_lgb@{h}: {compute_brier(time_values, event_values, global_lgb_oof[h], h):.6f}")
    print(f"glob_cat@{h}: {compute_brier(time_values, event_values, global_cat_oof[h], h):.6f}")
    print(f"linear@{h}:   {compute_brier(time_values, event_values, linear_oof[h], h):.6f}")
    print(f"near_lgb@{h}: {compute_brier(time_values, event_values, near_lgb_oof[h], h):.6f}")

for h in [24, 48]:
    far_lgb_oof[h], far_lgb_test[h] = repeated_binary_model(
        X_far_lgb_train,
        X_far_lgb_test,
        y_map[h],
        mask_map[h],
        make_lgb_builder(far_lgb_cfgs[h]),
        TREE_SEEDS,
        desired_splits=5,
        horizon=h,
        use_ipcw=True,
    )
    far_lgb_oof[h] = stabilize(far_lgb_oof[h], anchor_oof[h], 0.94)
    far_lgb_test[h] = stabilize(far_lgb_test[h], anchor_test[h], 0.94)
    print(f"far_lgb@{h}:  {compute_brier(time_values, event_values, far_lgb_oof[h], h):.6f}")

cox_risk_oof, cox_risk_test = repeated_xgb_cox_risk(
    X_surv_train,
    X_surv_test,
    time_values,
    event_values,
    SURV_SEEDS,
    cox_cfgs,
    anchor_signal_train,
    anchor_signal_test,
    desired_splits=5,
)

aft_risk_oof, aft_risk_test = repeated_xgb_aft_risk(
    X_surv_train,
    X_surv_test,
    time_values,
    event_values,
    SURV_SEEDS,
    aft_cfgs,
    anchor_signal_train,
    anchor_signal_test,
    desired_splits=5,
)

cox_oof, cox_test = {}, {}
aft_oof, aft_test = {}, {}

for h in [12, 24, 48]:
    cox_oof[h], cox_test[h] = repeated_isotonic(
        cox_risk_oof,
        cox_risk_test,
        y_map[h],
        mask_map[h],
        ANCHOR_SEEDS,
        desired_splits=5,
    )
    aft_oof[h], aft_test[h] = repeated_isotonic(
        aft_risk_oof,
        aft_risk_test,
        y_map[h],
        mask_map[h],
        ANCHOR_SEEDS,
        desired_splits=5,
    )
    cox_oof[h] = stabilize(cox_oof[h], anchor_oof[h], 0.95 if h == 48 else 0.94)
    cox_test[h] = stabilize(cox_test[h], anchor_test[h], 0.95 if h == 48 else 0.94)
    aft_oof[h] = stabilize(aft_oof[h], anchor_oof[h], 0.96 if h == 48 else 0.95)
    aft_test[h] = stabilize(aft_test[h], anchor_test[h], 0.96 if h == 48 else 0.95)

    print(f"cox@{h}:      {compute_brier(time_values, event_values, cox_oof[h], h):.6f}")
    print(f"aft@{h}:      {compute_brier(time_values, event_values, aft_oof[h], h):.6f}")

base_oof = {
    "anchor": anchor_oof,
    "phys": phys_oof,
    "glob_lgb": global_lgb_oof,
    "glob_cat": global_cat_oof,
    "lin": linear_oof,
    "cox": cox_oof,
    "aft": aft_oof,
}
base_test = {
    "anchor": anchor_test,
    "phys": phys_test,
    "glob_lgb": global_lgb_test,
    "glob_cat": global_cat_test,
    "lin": linear_test,
    "cox": cox_test,
    "aft": aft_test,
}

near_lgb_store = {"oof": near_lgb_oof, "test": near_lgb_test}
far_lgb_store = {"oof": far_lgb_oof, "test": far_lgb_test}


def assemble_available(h, zone, split_kind):
    store = base_oof if split_kind == "oof" else base_test
    avail = {}
    for key in ["anchor", "phys", "glob_lgb", "glob_cat", "lin", "cox", "aft"]:
        if h in store[key]:
            avail[key] = store[key][h]
    if zone == "near":
        if h in near_lgb_store[split_kind]:
            avail["zone_lgb"] = near_lgb_store[split_kind][h]
    else:
        if h in far_lgb_store[split_kind]:
            avail["zone_lgb"] = far_lgb_store[split_kind][h]
    names = list(avail.keys())
    P = np.column_stack([avail[name] for name in names])
    return names, P


GATE_CFG = {
    12: (4500.0, 1400.0),
    24: (5200.0, 1800.0),
    48: (6500.0, 2500.0),
}

weights_log = {}
oof_final = np.zeros((len(train_df), 4), dtype=float)
test_final = np.zeros((len(test_df), 4), dtype=float)

for col_idx, h in enumerate([12, 24, 48]):
    center, width = GATE_CFG[h]
    gate_train = soft_gate(dist_train, center, width)
    gate_test = soft_gate(dist_test, center, width)

    names_n, Pn_oof = assemble_available(h, "near", "oof")
    _, Pn_test = assemble_available(h, "near", "test")
    near_mask = mask_map[h] & (dist_train < 5000.0)
    prior_n = build_data_prior(Pn_oof, y_map[h], names_n, near_mask)
    reg_n, shrink_n = zone_blend_cfg(h, "near")
    if near_mask.sum() >= max(10, len(names_n) + 4):
        w_n, opt_n = optimize_blend(Pn_oof[near_mask], y_map[h][near_mask], prior_n, reg_n, shrink_n)
    else:
        w_n, opt_n = prior_n.copy(), prior_n.copy()
    pred_near_oof = np.clip(Pn_oof @ w_n, 0.0, 1.0)
    pred_near_test = np.clip(Pn_test @ w_n, 0.0, 1.0)

    names_f, Pf_oof = assemble_available(h, "far", "oof")
    _, Pf_test = assemble_available(h, "far", "test")
    far_mask = mask_map[h] & ~(dist_train < 5000.0)
    prior_f = build_data_prior(Pf_oof, y_map[h], names_f, far_mask)
    reg_f, shrink_f = zone_blend_cfg(h, "far")
    if far_mask.sum() >= max(10, len(names_f) + 4):
        w_f, opt_f = optimize_blend(Pf_oof[far_mask], y_map[h][far_mask], prior_f, reg_f, shrink_f)
    else:
        w_f, opt_f = prior_f.copy(), prior_f.copy()
    pred_far_oof = np.clip(Pf_oof @ w_f, 0.0, 1.0)
    pred_far_test = np.clip(Pf_test @ w_f, 0.0, 1.0)

    oof_final[:, col_idx] = gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof
    test_final[:, col_idx] = gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test

    weights_log[f"{h}_near"] = {
        "names": names_n,
        "prior": [float(x) for x in prior_n],
        "opt": [float(x) for x in opt_n],
        "final": [float(x) for x in w_n],
    }
    weights_log[f"{h}_far"] = {
        "names": names_f,
        "prior": [float(x) for x in prior_f],
        "opt": [float(x) for x in opt_f],
        "final": [float(x) for x in w_f],
    }

consensus_train = 0.55 * normalize_rank(cox_risk_oof) + 0.45 * normalize_rank(aft_risk_oof)
consensus_test = 0.55 * normalize_rank(cox_risk_test) + 0.45 * normalize_rank(aft_risk_test)

rank_alpha = {12: 0.02, 24: 0.04, 48: 0.06}
rank_gate = {12: (4500.0, 1600.0), 24: (5500.0, 2200.0), 48: (7000.0, 2800.0)}

for col_idx, h in enumerate([12, 24, 48]):
    center, width = rank_gate[h]
    local_alpha_train = rank_alpha[h] * (0.60 + 0.40 * soft_gate(dist_train, center, width))
    local_alpha_test = rank_alpha[h] * (0.60 + 0.40 * soft_gate(dist_test, center, width))

    rb_oof = rank_to_distribution(oof_final[:, col_idx], consensus_train)
    rb_test = rank_to_distribution(test_final[:, col_idx], consensus_test)

    oof_final[:, col_idx] = (1.0 - local_alpha_train) * oof_final[:, col_idx] + local_alpha_train * rb_oof
    test_final[:, col_idx] = (1.0 - local_alpha_test) * test_final[:, col_idx] + local_alpha_test * rb_test

oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0

oof_final = np.nan_to_num(oof_final, nan=0.0, posinf=1.0, neginf=0.0)
test_final = np.nan_to_num(test_final, nan=0.0, posinf=1.0, neginf=0.0)

oof_final = enforce_monotonicity(oof_final)
test_final = enforce_monotonicity(test_final)
oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0

metrics = compute_hybrid_score(time_values, event_values, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3])
print("OOF Hybrid =", round(metrics["hybrid"], 6))
print("OOF C-Index =", round(metrics["c_index"], 6))
print("OOF Weighted Brier =", round(metrics["weighted_brier"], 6))
print("OOF Brier@24 =", round(metrics["brier24"], 6))
print("OOF Brier@48 =", round(metrics["brier48"], 6))
print("OOF Brier@72 =", round(metrics["brier72"], 6))
print("Train distance split:",
      int((dist_train < 5000.0).sum()), "near /", int((dist_train >= 5000.0).sum()), "far")
print("Test distance split:",
      int((dist_test < 5000.0).sum()), "near /", int((dist_test >= 5000.0).sum()), "far")

submission = pd.DataFrame(
    {
        "event_id": test_df["event_id"].values,
        "prob_12h": test_final[:, 0],
        "prob_24h": test_final[:, 1],
        "prob_48h": test_final[:, 2],
        "prob_72h": test_final[:, 3],
    }
)

submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")
validate_submission(submission, sample_sub)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

oof_dump = pd.DataFrame(
    {
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_final[:, 0],
        "prob_24h": oof_final[:, 1],
        "prob_48h": oof_final[:, 2],
        "prob_72h": oof_final[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    }
)
oof_dump.to_csv(os.path.join(WORK_DIR, "oof_predictions.csv"), index=False)

with open(os.path.join(WORK_DIR, "blend_weights.json"), "w") as f:
    json.dump(weights_log, f, indent=2)

print("Saved:", OUTPUT_PATH)
print(submission.head())


DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN = (221, 37) TEST = (95, 35)
anchor@12: 0.069100
phys@12:   0.077625
anchor@24: 0.029499
phys@24:   0.040685
anchor@48: 0.018980
phys@48:   0.034266
glob_lgb@12: 0.057311
glob_cat@12: 0.056903
linear@12:   0.056906
near_lgb@12: 0.056316
glob_lgb@24: 0.027960
glob_cat@24: 0.030215
linear@24:   0.031520
near_lgb@24: 0.027837
glob_lgb@48: 0.016747
glob_cat@48: 0.019301
linear@48:   0.019085
near_lgb@48: 0.015379
far_lgb@24:  0.027803
far_lgb@48:  0.014556
cox@12:      0.057619
aft@12:      0.063655
cox@24:      0.036090
aft@24:      0.034381
cox@48:      0.026332
aft@48:      0.023866
OOF Hybrid = 0.969299
OOF C-Index = 0.933919
OOF Weighted Brier = 0.015538
OOF Brier@24 = 0.028793
OOF Brier@48 = 0.017249
OOF Brier@72 = 0.0
Train distance split: 69 near / 152 far
Test distance split: 28 near / 67 far
Saved: /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.001554  0.0052